# SECAI — Data Cleaning
Source: `timesheet_output.xlsx` → sheet `All Records`

Each cell tackles one specific issue found during initial screening.

In [ ]:
import re
import pandas as pd

FILE_PATH  = "timesheet_output.xlsx"
SHEET_NAME = "All Records"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df.columns = df.columns.str.strip()
for col in df.columns:
    df[col] = df[col].str.strip()

print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

## Fix 0a · Date audit — one row per page
Pages should run in date order but are not guaranteed to.
This cell shows the raw date and day for each page so problems are visible
before any corrections are applied.

In [ ]:
# One representative row per page, sorted by page number
page_summary = (
    df[["Page", "Date", "Day"]]
    .drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg")
    .drop(columns="_pg")
    .reset_index(drop=True)
)

# Try to parse each date and flag anything that can't be parsed or has a bad year
VALID_YEARS = {25, 2025, 26, 2026}

def audit_date(v):
    if pd.isna(v) or str(v).strip() == "": return "⚠ blank"
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        d = pd.to_datetime(s, dayfirst=False)
        yr = d.year % 100  # normalise 2025→25, 1985→85
        full_yr = d.year
        if full_yr % 100 not in {25, 26}:
            return f"⚠ wrong year ({full_yr})"
        return f"OK — {d.strftime('%b %d, %Y')} ({d.day_name()})"
    except Exception:
        return "⚠ unparseable"

page_summary["Audit"] = page_summary["Date"].apply(audit_date)

try:
    import jinja2
    display(page_summary.style.apply(
        lambda r: ["background-color: #fdd" if "⚠" in str(r["Audit"]) else "" for _ in r],
        axis=1
    ))
except ImportError:
    display(page_summary)

## Fix 0b · Correct bad dates using page sequence context
Each correction is derived from the surrounding pages and cross-checked
against the recorded Day of week where available.
All corrections target year 2025 (YY = 25).

| Page | Raw value | Problem | Correction | Verified by |
|------|-----------|---------|------------|-------------|
| 4 | `11-04-25` | Month typo 11→12 | `12/04/25` | p3=12/03, p5=12/05 |
| 7 | `10/08/05` | Month 10→12, year 05→25 | `12/08/25` | p6=12/06, p8=12/09 |
| 9 | `12/10/2015` | Year 2015→25 | `12/10/25` | p8=12/09, p10=12/11 |
| 10 | `12/11/23` | Year 23→25 | `12/11/25` | p9=12/10, p11=12/12 |
| 11 | `4/10/05` | Month 4→12, year 05→25 | `12/12/25` | p10=12/11, Day=Friday ✓ |
| 12 | `12/12/85` | Year 85→25, day→13 | `12/13/25` | next Saturday after p11, Day blank |
| 16 | `14/19/2025` | Impossible date | `12/18/25` | Day=JUEVES=Thu, 12/18/25 is Thursday ✓ |
| 18 | `12/19/20` | Year 20→25 | `12/19/25` | p17 blank, p20=12/20, Day=Friday ✓ |
| 22 | `12/23/24` | Year 24→25 | `12/22/25` | p23=12/23, Day=Monday, 12/22/25 is Monday ✓ |

In [ ]:
# Each entry documents one date correction — field, raw value, fix, and reasoning
CORRECTIONS_LOG = [
    {
        "fix_id":   "F0b-01",
        "page":     "4",
        "field":    "Date",
        "raw":      "11-04-25",
        "fix":      "12/04/25",
        "problem":  "Month digit reads '11' but surrounding pages are all December. Likely OCR misread of '12'.",
        "evidence": "p3 = 12/03/25, p5 = 12/05/25. Sequence implies 12/04/25.",
    },
    {
        "fix_id":   "F0b-02",
        "page":     "7",
        "field":    "Date",
        "raw":      "10/08/05",
        "fix":      "12/08/25",
        "problem":  "Month reads '10' and year reads '05'. Both are OCR errors — digit '1' dropped from month, '2' dropped from year.",
        "evidence": "p6 = 12/06/25, p8 = 12/09/25. Day recorded as Monday; 12/08/25 is a Monday.",
    },
    {
        "fix_id":   "F0b-03",
        "page":     "9",
        "field":    "Date",
        "raw":      "12/10/2015",
        "fix":      "12/10/25",
        "problem":  "Year recorded as 2015 — extra '1' inserted, likely OCR artifact from adjacent text.",
        "evidence": "p8 = 12/09/25, p10 = 12/11/25. Day = Thursday; 12/10/25 is a Wednesday — Day will be fixed separately.",
    },
    {
        "fix_id":   "F0b-04",
        "page":     "10",
        "field":    "Date",
        "raw":      "12/11/23",
        "fix":      "12/11/25",
        "problem":  "Year reads '23' instead of '25'. Single digit transposition.",
        "evidence": "p9 = 12/10/25, p11 = 12/12/25. Day = Thursday; 12/11/25 is a Thursday.",
    },
    {
        "fix_id":   "F0b-05",
        "page":     "11",
        "field":    "Date",
        "raw":      "4/10/05",
        "fix":      "12/12/25",
        "problem":  "Date is fully garbled — month reads '4', year reads '05'. Entire value is an OCR misread.",
        "evidence": "p10 = 12/11/25 (Thursday), p12 = 12/13/25 (Saturday). p11 must be 12/12/25. Day = Friday; 12/12/25 is a Friday.",
    },
    {
        "fix_id":   "F0b-06",
        "page":     "12",
        "field":    "Date",
        "raw":      "12/12/85",
        "fix":      "12/13/25",
        "problem":  "Year reads '85' and day digit '3' was dropped. OCR misread of handwritten '25'.",
        "evidence": "p11 = 12/12/25 (Friday), p13 = 12/10/25. Next calendar day is Saturday 12/13/25. Day field is blank on this page.",
    },
    {
        "fix_id":   "F0b-07",
        "page":     "16",
        "field":    "Date",
        "raw":      "14/19/2025",
        "fix":      "12/18/25",
        "problem":  "Impossible date — month '14' and day '19' are both invalid. OCR merged adjacent characters.",
        "evidence": "p15 = 12/17/25, p18 = 12/19/25. Day = JUEVES (Thursday); 12/18/25 is a Thursday.",
    },
    {
        "fix_id":   "F0b-08",
        "page":     "18",
        "field":    "Date",
        "raw":      "12/19/20",
        "fix":      "12/19/25",
        "problem":  "Year reads '20' instead of '25'. Digit '5' misread as '0'.",
        "evidence": "p17 = blank, p20 = 12/20/25. Day = Friday; 12/19/25 is a Friday.",
    },
    {
        "fix_id":   "F0b-09",
        "page":     "22",
        "field":    "Date",
        "raw":      "12/23/24",
        "fix":      "12/22/25",
        "problem":  "Year reads '24' and day reads '23' — but p23 is also 12/23/25, so this page must be the day before.",
        "evidence": "p23 = 12/23/25 (Tuesday), p24 = 12/24/25. Day = Monday; 12/22/25 is a Monday.",
    },
]

# Apply corrections to df
total_fixed = 0
for entry in CORRECTIONS_LOG:
    mask = df["Page"] == entry["page"]
    n = int(mask.sum())
    df.loc[mask, entry["field"]] = entry["fix"]
    total_fixed += n
    parsed = pd.to_datetime(entry["fix"])
    print(f"  [{entry['fix_id']}] Page {entry['page']:>2s}: '{entry['raw']}'  →  '{entry['fix']}'  ({parsed.day_name()})  [{n} rows]")

print(f"\nTotal rows updated: {total_fixed}")

# Re-run audit to confirm all clear
print("\nDate audit after corrections:")
page_check = (
    df[["Page", "Date"]]
    .drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg").drop(columns="_pg")
    .reset_index(drop=True)
)
page_check["Audit"] = page_check["Date"].apply(audit_date)
remaining = page_check[page_check["Audit"].str.startswith("⚠")]
if remaining.empty:
    print("  All dates clean.")
else:
    print(f"  {len(remaining)} page(s) still need attention:")
    display(remaining)

## Fix 1 · Translate Spanish day names to English

In [ ]:
SPANISH_MAP = {
    "LUNES":      "Monday",
    "MARTES":     "Tuesday",
    "MIERCOLES":  "Wednesday",
    "MIÉRCOLES":  "Wednesday",
    "JUEVES":     "Thursday",
    "VIERNES":    "Friday",
    "SABADO":     "Saturday",
    "SÁBADO":     "Saturday",
    "DOMINGO":    "Sunday",
}

before = df["Day"].value_counts(dropna=False).to_dict()

df["Day"] = df["Day"].apply(
    lambda v: SPANISH_MAP.get(str(v).upper().strip(), v) if pd.notna(v) else v
)

after = df["Day"].value_counts(dropna=False)

# Summary of what changed
changed = {k: v for k, v in before.items() if str(k).upper().strip() in SPANISH_MAP}
print(f"Translated {sum(changed.values())} rows across {len(changed)} Spanish value(s):")
for spanish, count in changed.items():
    english = SPANISH_MAP[str(spanish).upper().strip()]
    print(f"  '{spanish}' ({count} rows)  →  '{english}'")

print("\nDistinct Day values after fix:")
display(after.rename_axis("Day").reset_index(name="Count"))

## Fix 1 · Check — rows where Day is still not a standard weekday
After the Spanish translation, any remaining non-standard value needs manual review.

In [ ]:
VALID_DAYS = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"}

mask = ~df["Day"].apply(lambda v: str(v).strip().title() in VALID_DAYS if pd.notna(v) else False)
odd_days = df[mask][["Source File", "Page", "Date", "Day", "Employee Name"]].reset_index(drop=True)

print(f"{len(odd_days)} rows with non-standard Day values. Showing up to 10:")
display(odd_days.head(10))

## Fix 2 · Correct typo — 'Today' → 'Tuesday'
Page 8 date is `12.09.25` = December 9, 2025 = **Tuesday**.
'Today' is a known OCR/handwriting misread of 'Tuesday' on that page.
After applying the fix, cross-check every row by deriving the expected weekday
from its Date and flagging any row where Day and Date disagree.

In [ ]:
TYPO_MAP = {
    "Today": "Tuesday",   # page 8 — 12.09.25 is a Tuesday
}

before = df["Day"].value_counts(dropna=False).to_dict()

df["Day"] = df["Day"].apply(
    lambda v: TYPO_MAP.get(str(v).strip(), v) if pd.notna(v) else v
)

fixed = {k: v for k, v in before.items() if str(k).strip() in TYPO_MAP}
print("Typo corrections applied:")
for typo, count in fixed.items():
    print(f"  '{typo}' ({count} rows)  →  '{TYPO_MAP[str(typo).strip()]}'")

# ── Date vs Day cross-check ──────────────────────────────────────────────────
# Parse dates with multiple separators (/ . -)
def safe_parse(v):
    if pd.isna(v) or str(v).strip() == "":
        return pd.NaT
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        return pd.to_datetime(s, dayfirst=False)
    except Exception:
        return pd.NaT

df["_parsed_date"]    = df["Date"].apply(safe_parse)
df["_expected_day"]   = df["_parsed_date"].apply(
    lambda d: d.day_name() if pd.notna(d) else None
)

# Only compare rows where both Day and Date are present and date parsed cleanly
comparable = df[
    df["Day"].notna() & df["Day"].ne("") &
    df["_expected_day"].notna()
].copy()

conflicts = comparable[
    comparable["Day"].str.strip().str.title() !=
    comparable["_expected_day"]
][["Source File", "Page", "Date", "_parsed_date", "_expected_day", "Day", "Employee Name"]]
conflicts = conflicts.rename(columns={"_parsed_date": "Date (parsed)", "_expected_day": "Expected Day"})
conflicts = conflicts.reset_index(drop=True)

print(f"\nDate vs Day conflicts: {len(conflicts)} rows")
display(conflicts)

## Fix 3 · Normalise time columns to H:MMam/pm format
Target format: `7:00AM`, `12:00PM`, `4:00PM` — no space between time and am/pm.

Variants found in the data across `Sched Start`, `Sched End`, `Actual Start`,
`Lunch Out`, `Lunch In`, `Actual End`:
- Spacing: `7 AM`, `7AM`, `7:00`, `7:00 AM`
- Case: `9am`, `12PM`, `12m`, `1pm`
- Typos: `12 Pu` (→ PM), `1 Pu` (→ PM), `4 Pu` (→ PM), `5 Pu` (→ PM)
- Truncated: `6:5 AM` (→ 6:05AM), `9:5` (→ 9:05)
- Noise: `7 AM AM` (→ 7:00AM)

In [ ]:
import re

TIME_COLS = ["Sched Start", "Sched End", "Actual Start", "Lunch Out", "Lunch In", "Actual End"]

# Known literal typo fixes applied before parsing
LITERAL_FIXES = {
    "12 Pu": "12:00PM",
    "1 Pu":  "1:00PM",
    "4 Pu":  "4:00PM",
    "5 Pu":  "5:00PM",
    "7 AM AM": "7:00AM",
    "12m":   "12:00PM",
    "12M":   "12:00PM",
}

def normalise_time(v):
    if pd.isna(v) or str(v).strip() == "":
        return v
    s = str(v).strip()

    # Apply literal fixes first
    if s in LITERAL_FIXES:
        return LITERAL_FIXES[s]

    # Normalise 'Pu' → 'PM' anywhere in string
    s = re.sub(r'\bPu\b', 'PM', s, flags=re.IGNORECASE)

    # Remove duplicate am/pm suffixes e.g. '7 AM AM'
    s = re.sub(r'(AM|PM)(\s*(AM|PM))+', r'\1', s, flags=re.IGNORECASE)

    # Extract components with regex
    # Handles: 7, 7AM, 7 AM, 7:00, 7:00AM, 7:00 AM, 6:5 AM, 2:30AM
    m = re.match(
        r'^(\d{1,2})(?::(\d{1,2}))?\s*(AM|PM|am|pm)?$',
        s.strip()
    )
    if not m:
        return v  # unparseable — return original so it stays visible

    hour    = int(m.group(1))
    minutes = m.group(2)
    suffix  = m.group(3)

    # Pad single-digit minutes: '6:5' → '6:05'
    if minutes is None:
        minutes = "00"
    elif len(minutes) == 1:
        minutes = minutes + "0"

    # Infer AM/PM from hour if missing
    if suffix is None:
        if hour == 0 or hour == 12:
            suffix = "PM"
        elif 1 <= hour <= 6:
            suffix = "AM"   # 1–6 without suffix → early morning (e.g. 2:00AM shift end)
        else:
            suffix = "AM"   # 7–11 → morning

    return f"{hour}:{minutes}{suffix.upper()}"

# Apply to all time columns and report
for col in TIME_COLS:
    before = df[col].copy()
    df[col] = df[col].apply(normalise_time)
    changed = (before.fillna("") != df[col].fillna("")).sum()
    print(f"  {col:<14} — {changed} cells updated")

print("\nDistinct values after normalisation:")
for col in TIME_COLS:
    vals = df[col].value_counts(dropna=False)
    print(f"\n  {col}: {dict(vals)}")

## Fix 5 · Standardise Date format to MM/DD/YY
After date value corrections in Fix 0b, some dates still use `-` or `.` separators.
This cell normalises all dates to a consistent `MM/DD/YY` format.

In [ ]:
def format_date(v):
    if pd.isna(v) or str(v).strip() == "":
        return v
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        d = pd.to_datetime(s, dayfirst=False)
        return d.strftime("%-m/%-d/%y") if hasattr(d, 'strftime') else d.strftime("%m/%d/%y").lstrip("0").replace("/0", "/")
    except Exception:
        return v

# Use platform-safe formatting
import platform
def format_date(v):
    if pd.isna(v) or str(v).strip() == "":
        return v
    s = str(v).strip().replace(".", "/").replace("-", "/")
    try:
        d = pd.to_datetime(s, dayfirst=False)
        return f"{d.month}/{d.day}/{str(d.year)[-2:]}"
    except Exception:
        return v

before = df["Date"].copy()
df["Date"] = df["Date"].apply(format_date)
changed = (before.fillna("") != df["Date"].fillna("")).sum()
print(f"Date format standardised: {changed} cells updated")
print("\nDistinct Date values after fix:")
display(
    df[["Page", "Date"]].drop_duplicates(subset=["Page"])
    .assign(_pg=lambda d: pd.to_numeric(d["Page"], errors="coerce"))
    .sort_values("_pg").drop(columns="_pg")
    .reset_index(drop=True)
)

## Fix 6 · Fill missing Date, Day, Business Unit, Project from previous page
Pages 17, 19, and 21 are continuation pages (management staff only) with no header data.
They inherit everything from the page directly before them.

Business Unit and Project also have OCR variants across other pages that are corrected here:
- `Business Unit`: all variants of `107871.5` normalised to `107871.5`
- `Project`: OCR misreads of `Final Construction Clean - Ceiling Project 2` corrected

In [ ]:
# ── Step 1: Fix Business Unit OCR variants ───────────────────────────────────
BU_CORRECT = "107871.5"
BU_VARIANTS = {
    "1078715", "107071-S", "107371-5", "10787.1.5",
    "107871-5", "10787L5", "107871 5", "10787LS"
}
bu_before = df["Business Unit"].copy()
df["Business Unit"] = df["Business Unit"].apply(
    lambda v: BU_CORRECT if str(v).strip() in BU_VARIANTS else v
)
bu_fixed = (bu_before.fillna("") != df["Business Unit"].fillna("")).sum()
print(f"Business Unit: {bu_fixed} cells corrected to '{BU_CORRECT}'")

# ── Step 2: Fix Project OCR variants ────────────────────────────────────────
PROJ_CORRECT = "Final Construction Clean - Ceiling Project 2"
PROJ_VARIANTS = {
    "Final Construction Clean - Online Project 2",
    "Final Construction Clean - CeFind Project 2",
    "Final Construction Clean - Ceiling Project 7",
    "Final Construction Clean - Gating Project 2",
    "Final Construction Clean - Ceiling Project 3",
    "Final Construction Clean - Ceiling Project 1",
    "Fruit Construction Clean - Ceiling Project 2",
}
proj_before = df["Project"].copy()
df["Project"] = df["Project"].apply(
    lambda v: PROJ_CORRECT if str(v).strip() in PROJ_VARIANTS else v
)
proj_fixed = (proj_before.fillna("") != df["Project"].fillna("")).sum()
print(f"Project:       {proj_fixed} cells corrected to '{PROJ_CORRECT}'")

# ── Step 3: Fill missing header fields from previous page ───────────────────
FILL_COLS = ["Date", "Day", "Business Unit", "Project"]

# Build a page → values map from rows that have data
page_vals = (
    df[FILL_COLS + ["Page"]]
    .dropna(subset=FILL_COLS, how="all")
    .drop_duplicates(subset=["Page"])
    .set_index("Page")
)

# Sort pages numerically so we can look back at the previous page
sorted_pages = sorted(df["Page"].dropna().unique(), key=lambda x: int(x) if str(x).isdigit() else 999)

fill_count = 0
for i, page in enumerate(sorted_pages):
    mask = df["Page"] == page
    for col in FILL_COLS:
        # Only fill if ALL rows on this page are blank for this column
        if df.loc[mask, col].isna().all() or df.loc[mask, col].eq("").all():
            # Look back through previous pages for a value
            for prev_page in reversed(sorted_pages[:i]):
                if prev_page in page_vals.index and pd.notna(page_vals.at[prev_page, col]):
                    fill_val = page_vals.at[prev_page, col]
                    df.loc[mask, col] = fill_val
                    fill_count += df.loc[mask, col].shape[0]
                    print(f"  Page {page} · {col}: filled from page {prev_page} → '{fill_val}'")
                    break

print(f"\nTotal cells filled from previous page: {fill_count}")

## Fix 4 · Reorder columns and add Recorded Hours
New column order: `Source File`, `Page`, `Date`, `Day`, `Business Unit`, `Project`,
`EIN`, `Employee Name`, `Staff Type`, `Job Task`, `Title`, `Sched Start`, `Sched End`,
`Sched Hours`, `Actual Start`, `Lunch Out`, `Lunch In`, `Actual End`, `Actual Hours`,
`Recorded Hours`, `Absent`, `Schedule Changed`, `Confidence`, `Status`

`Recorded Hours` is a copy of `Actual Hours` at this point in the cleaning pipeline —
a snapshot before any billing adjustments are applied downstream.

In [ ]:
# Add Recorded Hours as a copy of Actual Hours
df["Recorded Hours"] = df["Actual Hours"]

# Define new column order
NEW_ORDER = [
    "Source File", "Page", "Date", "Day",
    "Business Unit", "Project",
    "EIN", "Employee Name", "Title", "Recorded Hours",
    "Staff Type", "Job Task",
    "Sched Start", "Sched End", "Sched Hours",
    "Actual Start", "Lunch Out", "Lunch In", "Actual End",
    "Actual Hours",
    "Absent", "Schedule Changed", "Confidence", "Status",
]

# Safety check — catch any column in df not accounted for
base_cols = [c for c in df.columns if not c.startswith("_")]
missing_from_order = [c for c in base_cols if c not in NEW_ORDER]
if missing_from_order:
    print(f"⚠ Columns not in NEW_ORDER (will be appended at end): {missing_from_order}")
    NEW_ORDER += missing_from_order

df = df[[c for c in NEW_ORDER if c in df.columns]]

print("Column order after fix:")
for i, col in enumerate(df.columns, 1):
    marker = "← new" if col == "Recorded Hours" else "← moved" if col in ("Title", "EIN") else ""
    print(f"  {i:>2}. {col}  {marker}")

---
## Export · Generate cleaned Excel file
Drops internal helper columns, highlights every changed cell in yellow,
and adds a comment on each showing the original value.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

OUT_PATH = "secai_cleaned.xlsx"

# Reload original to diff against
df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, dtype=str)
df_raw.columns = df_raw.columns.str.strip()
for col in df_raw.columns:
    df_raw[col] = df_raw[col].str.strip()

# Drop internal helper columns before export
export_cols = [c for c in df.columns if not c.startswith("_")]
df_export   = df[export_cols].copy()

# Write clean data
df_export.to_excel(OUT_PATH, index=False, sheet_name="Cleaned")

# Open with openpyxl to apply formatting
wb = load_workbook(OUT_PATH)
ws = wb["Cleaned"]

# Map column name → Excel column index
col_idx = {cell.value: cell.column for cell in ws[1]}

YELLOW = PatternFill("solid", fgColor="FFF3CD")
BOLD   = Font(bold=True)

changed_count = 0
for col in export_cols:
    if col not in df_raw.columns or col not in col_idx:
        continue
    for i in df_export.index:
        old = "" if pd.isna(df_raw.at[i, col]) else str(df_raw.at[i, col]).strip()
        new = "" if pd.isna(df_export.at[i, col]) else str(df_export.at[i, col]).strip()
        if old != new:
            cell = ws.cell(row=i + 2, column=col_idx[col])
            cell.fill = YELLOW
            c = Comment(f"Original: {old}", "SECAI")
            c.width, c.height = 180, 40
            cell.comment = c
            changed_count += 1

# Bold header, freeze top row, auto-width columns
for cell in ws[1]:
    cell.font = BOLD
ws.freeze_panes = "A2"
for col_cells in ws.columns:
    width = max((len(str(c.value)) if c.value else 0) for c in col_cells)
    ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(width + 2, 30)

wb.save(OUT_PATH)
print(f"Saved → {OUT_PATH}")
print(f"  Rows          : {len(df_export):,}")
print(f"  Columns       : {len(export_cols)}")
print(f"  Changed cells : {changed_count}")